# Cleaner Colab GPU Prototype

앱 없이 Python만으로 `사진 -> 라벨 -> CLIP embedding -> 개인화 모델 -> 삭제 후보 추천` 루프를 검증하는 노트북입니다.

중요: 이 파일을 VS Code에서 실행하면 로컬 PC 커널이 사용됩니다. Colab GPU를 쓰려면 이 `.ipynb`를 Google Colab 웹사이트에서 열고 실행해야 합니다.

Colab에서 먼저 `런타임 -> 런타임 유형 변경 -> 하드웨어 가속기 -> T4 GPU`를 선택하세요.

In [ ]:
from pathlib import Path
import os
import sys

# If this notebook is opened from the repository root, this is enough.
# In Colab, upload/clone the repo to /content/Cleaner and uncomment the next line.
# %cd /content/Cleaner

ROOT = Path.cwd()
if not (ROOT / 'src' / 'cleaner').exists() and (Path('/content/Cleaner/src/cleaner')).exists():
    os.chdir('/content/Cleaner')
    ROOT = Path.cwd()

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
if not IN_COLAB:
    print('WARNING: This notebook is running locally, not on Google Colab.')
    print('VS Code cannot use Colab hosted GPU just by opening this local .ipynb file.')
    print('Open this notebook at https://colab.research.google.com/ to use Colab GPU.')

sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)


In [ ]:
%pip install -q -r requirements-colab.txt


In [ ]:
import torch

os.environ['CLEANER_DATA_DIR'] = '/content/cleaner_data'
os.environ['CLEANER_DEVICE'] = 'auto'
os.environ['CLEANER_BATCH_SIZE'] = '16'

print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU is attached to this kernel.')
    print('If you are in VS Code, this is your local kernel. Open the notebook in Colab and select a GPU runtime.')


## 1. 데모 데이터로 전체 파이프라인 확인

In [ ]:
!python scripts/create_demo_photos.py --out demo_photos --count 120
!python scripts/create_demo_labels.py --photo-dir demo_photos --data-dir /content/cleaner_data --max 80
!python scripts/run_pipeline.py --photo-dir demo_photos --data-dir /content/cleaner_data --max-embeddings 120 --top-n 20 --include-labeled


In [ ]:
import pandas as pd

candidates = pd.read_csv('/content/cleaner_data/delete_candidates.csv')
candidates.head(20)


## 2. 실제 사진 일부로 실험

`/content/my_photos` 폴더를 만들고 사진 일부만 업로드한 뒤 실행하세요.

In [ ]:
PHOTO_DIR = '/content/my_photos'
Path(PHOTO_DIR).mkdir(parents=True, exist_ok=True)
print('Upload sample photos to:', PHOTO_DIR)


### 방법 A: 노트북 버튼 라벨러

버튼 의미: `Keep=1`, `Discard=0`, `Important=1 weight 2.0`, `Later=-1`.

In [ ]:
from cleaner.config import get_config
from cleaner.notebook import start_labeler

cfg = get_config()
# start_labeler(PHOTO_DIR, cfg, limit=100)


### 방법 B: CSV로 라벨 작성

In [ ]:
!python scripts/export_label_template.py --photo-dir /content/my_photos --data-dir /content/cleaner_data --out /content/label_template.csv --max 200
print('Fill /content/label_template.csv, then run the import cell below.')


In [ ]:
# After editing /content/label_template.csv:
# !python scripts/import_labels.py /content/label_template.csv --data-dir /content/cleaner_data


## 3. 실제 사진 파이프라인 실행

In [ ]:
!python scripts/run_pipeline.py --photo-dir /content/my_photos --data-dir /content/cleaner_data --top-n 50


In [ ]:
pd.read_csv('/content/cleaner_data/delete_candidates.csv').head(50)
